In [12]:
import pandas as pd
import numpy as np
from datetime import datetime, timezone


In [13]:
ARTIFACTS_DIR = "artifacts"
OUTPUT_DIR = "output"

forecast = pd.read_csv(f"../{ARTIFACTS_DIR}/weekly_forecast_future_all_skus.csv")
inventory = pd.read_csv(f"../{OUTPUT_DIR}/inventory_events.csv")
vendors = pd.read_csv(f"../{OUTPUT_DIR}/vendors.csv")
cod = pd.read_csv(f"../{ARTIFACTS_DIR}/cod_intelligence.csv")

forecast["week"] = pd.to_datetime(forecast["week"])


In [4]:
SERVICE_LEVEL_BY_RISK = {
    "LOW": "P50",
    "MEDIUM": "P75",
    "HIGH": "P90",
}

DEFAULT_SERVICE_LEVEL = "P75"


In [14]:
latest_inventory = (
    inventory.sort_values("order_week")
    .groupby(["warehouse_id", "sku_id"])
    .tail(1)
    .rename(columns={"ending_stock": "on_hand"})
)

latest_inventory = latest_inventory[
    ["warehouse_id", "sku_id", "on_hand"]
]


In [15]:
df = (
    forecast
    .merge(latest_inventory, on="sku_id", how="left")
    .merge(vendors, on="sku_id", how="left")
)

df["on_hand"] = df["on_hand"].fillna(0)
df["lead_time_days"] = df["lead_time_days"].fillna(14)
df["MOQ"] = df["MOQ"].fillna(100)


In [16]:
df["lead_time_weeks"] = np.ceil(
    df["lead_time_days"] / 7
).astype(int)


In [17]:
df["lead_time_demand"] = df["p50"] * df["lead_time_weeks"]


In [18]:
df["demand_uncertainty"] = df["p90"] - df["p50"]


In [19]:
df["safety_stock"] = (
    df["demand_uncertainty"] * df["lead_time_weeks"]
)


In [20]:
df["reorder_point"] = (
    df["lead_time_demand"] + df["safety_stock"]
)


In [21]:
df["raw_reorder_qty"] = (
    df["reorder_point"] - df["on_hand"]
).clip(lower=0)


In [22]:
df["recommended_reorder_qty"] = df.apply(
    lambda r: 0 if r["raw_reorder_qty"] == 0
    else max(r["raw_reorder_qty"], r["MOQ"]),
    axis=1
)


In [24]:
# Build weekly_demand for trend calc

orders = pd.read_csv("../output/orders.csv")
orders["order_date"] = pd.to_datetime(orders["order_date"])

orders["order_week"] = (
    orders["order_date"]
    .dt.to_period("W")
    .apply(lambda p: p.start_time)
)

weekly_demand = (
    orders.groupby(["sku_id", "order_week"])
    .size()
    .reset_index(name="demand")
    .sort_values("order_week")
)


In [25]:
HIST_WINDOW = 26
RECENT_WINDOW = 8

trend_map = {}

for sku, hist in weekly_demand.groupby("sku_id"):
    if len(hist) < RECENT_WINDOW:
        trend_map[sku] = "INSUFFICIENT_DATA"
        continue

    recent_mean = hist.tail(RECENT_WINDOW)["demand"].mean()
    long_mean = hist.tail(HIST_WINDOW)["demand"].mean()

    if recent_mean > 1.1 * long_mean:
        trend_map[sku] = "UP"
    elif recent_mean < 0.9 * long_mean:
        trend_map[sku] = "DOWN"
    else:
        trend_map[sku] = "STABLE"

df["demand_trend"] = df["sku_id"].map(trend_map)


In [26]:
RUN_DATE = datetime.now(timezone.utc).date()

final_reorder_table = df[[
    "sku_id",
    "warehouse_id",
    "on_hand",
    "p50",
    "p90",
    "demand_uncertainty",
    "lead_time_weeks",
    "lead_time_demand",
    "safety_stock",
    "reorder_point",
    "recommended_reorder_qty",
    "MOQ",
    "demand_trend",
]]

final_reorder_table["run_date"] = RUN_DATE

final_reorder_table = final_reorder_table.sort_values(
    ["demand_trend", "recommended_reorder_qty"],
    ascending=[True, False]
)

final_reorder_table.head(20)


C:\Users\ACER\AppData\Local\Temp\ipykernel_22016\677218069.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_reorder_table["run_date"] = RUN_DATE


,sku_id,warehouse_id,on_hand,p50,p90,demand_uncertainty,lead_time_weeks,lead_time_demand,safety_stock,reorder_point,recommended_reorder_qty,MOQ,demand_trend,run_date
2286,SKU0022,south,101,134.167989,207.890944,73.722955,3,402.503966,221.168865,623.672831,522.672831,200,DOWN,2025-12-26
2230,SKU0022,south,101,139.916161,207.098562,67.182401,3,419.748483,201.547202,621.295685,520.295685,200,DOWN,2025-12-26
2226,SKU0022,south,101,135.966463,202.526006,66.559543,3,407.899388,199.678628,607.578017,506.578017,200,DOWN,2025-12-26
2266,SKU0022,south,101,130.244059,202.021713,71.777655,3,390.732176,215.332964,606.065140,505.065140,200,DOWN,2025-12-26
2270,SKU0022,south,101,128.890498,201.086431,72.195933,3,386.671494,216.587799,603.259293,502.259293,200,DOWN,2025-12-26
2392,SKU0024,north,267,110.797409,141.074768,30.277359,2,221.594818,60.554718,282.149536,500.000000,500,DOWN,2025-12-26
2393,SKU0024,south,100,110.797409,141.074768,30.277359,2,221.594818,60.554718,282.149536,500.000000,500,DOWN,2025-12-26
2395,SKU0024,east,253,110.797409,141.074768,30.277359,2,221.594818,60.554718,282.149536,500.000000,500,DOWN,2025-12-26
2396,SKU0024,north,267,134.751077,165.023394,30.272318,2,269.502153,60.544635,330.046788,500.000000,500,DOWN,2025-12-26
2397,SKU0024,south,100,134.751077,165.023394,30.272318,2,269.502153,60.544635,330.046788,500.000000,500,DOWN,2025-12-26


In [28]:
final_reorder_table.to_csv(
    f"{ARTIFACTS_DIR}/../reorder_recommendations.csv",
    index=False
)

print("Saved reorder_recommendations.csv")


Saved reorder_recommendations.csv
